# Sistema de Detecção de Fraude

Pipeline de detecção de fraude em transações de cartão de crédito. Este notebook importa a lógica de `fraud_ml/` (mesmo pacote usado por `MlFraude.py`) em vez de duplicá-la — assim script e notebook nunca ficam fora de sincronia.

In [1]:
import sys
import json
from pathlib import Path

if ".." not in sys.path:
    sys.path.append(str(Path.cwd().parent))

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from fraud_ml import config, data, evaluation, features, modeling

## 1. Carregar dados

In [2]:
df = data.load_training_data()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 144352 entries, 0 to 144351
Data columns (total 26 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   ssn         144352 non-null  str    
 1   cc_num      144352 non-null  int64  
 2   first       144352 non-null  str    
 3   last        144352 non-null  str    
 4   gender      144352 non-null  str    
 5   street      144352 non-null  str    
 6   city        144352 non-null  str    
 7   state       144352 non-null  str    
 8   zip         1468 non-null    float64
 9   lat         144352 non-null  float64
 10  long        144352 non-null  float64
 11  city_pop    144352 non-null  int64  
 12  job         144352 non-null  str    
 13  dob         144352 non-null  str    
 14  acct_num    144352 non-null  int64  
 15  profile     144352 non-null  str    
 16  trans_num   144194 non-null  str    
 17  trans_date  144194 non-null  str    
 18  trans_time  144194 non-null  str    
 19  unix_time   1

## 2. Limpeza e seleção de colunas

In [3]:
df = df.dropna(subset=config.TARGET_COLUMN)
y = df[config.TARGET_COLUMN]

df_train = df.drop(columns=[*config.PII_AND_HIGH_CARDINALITY_COLUMNS, config.TARGET_COLUMN])
df_train.info()

<class 'pandas.DataFrame'>
Index: 144194 entries, 0 to 144351
Data columns (total 11 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   gender      144194 non-null  str    
 1   city        144194 non-null  str    
 2   state       144194 non-null  str    
 3   city_pop    144194 non-null  int64  
 4   job         144194 non-null  str    
 5   dob         144194 non-null  str    
 6   profile     144194 non-null  str    
 7   trans_date  144194 non-null  str    
 8   trans_time  144194 non-null  str    
 9   category    144194 non-null  str    
 10  amt         144194 non-null  float64
dtypes: float64(1), int64(1), str(9)
memory usage: 13.2 MB


## 3. Engenharia de features

In [4]:
features.add_datetime_features(df_train)
features.add_age_feature(df_train)
df_train.head()

,gender,city,state,city_pop,job,profile,category,amt,hour,minute,seconds,year,month,day,age
0,M,Murrieta,CA,115971,Counselling psychologist,adults_2550_male_urban.json,misc_pos,2.27,15,23,53,2023,7,23,47
1,F,Ontario,CA,166856,"Nurse, mental health",adults_50up_female_urban.json,home,10.52,22,13,50,2023,12,13,61
2,M,Lancaster,CA,182305,Water quality scientist,adults_2550_male_urban.json,misc_pos,149.01,23,46,47,2023,7,23,40
3,F,Carson City,NV,61818,Health and safety inspector,adults_50up_female_urban.json,personal_care,32.01,16,45,57,2024,1,29,56
4,M,Rochester,NY,478127,Soil scientist,adults_50up_male_urban.json,misc_pos,137.31,4,16,14,2023,6,18,65


## 4. Encoding categórico

In [5]:
encoders = features.fit_label_encoders(df_train, config.CATEGORICAL_COLUMNS)
df_train[config.CATEGORICAL_COLUMNS].head()

,gender,job,profile,category,city,state
0,1,43,3,9,114,4
1,0,95,5,6,126,4
2,1,170,3,9,91,4
3,0,71,5,10,24,26
4,1,144,7,9,142,27


## 5. Split treino/teste e imputação

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    df_train, y, test_size=config.TEST_SIZE, random_state=config.RANDOM_SEED, stratify=y
)

imputer = SimpleImputer()
X_train_transform = imputer.fit_transform(X_train)
X_test_transform = imputer.transform(X_test)

target_encoder = LabelEncoder()
y_train_transformed = target_encoder.fit_transform(y_train)
y_test_transform = target_encoder.transform(y_test)

## 6. Comparação de modelos baseline

In [7]:
models = modeling.build_models(config.RANDOM_SEED)
baseline_scores = evaluation.fit_and_score(
    models, X_train_transform, X_test_transform, y_train_transformed, y_test_transform
)
print(json.dumps(baseline_scores, indent=2))

/Users/felipevieira/Developer/Transaction-Fraud-ML/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


{
  "KNN": {
    "accuracy": 0.9913311834668331,
    "precision_fraud": 0.6728395061728395,
    "recall_fraud": 0.3562091503267974,
    "f1_fraud": 0.4658119658119658,
    "roc_auc": 0.840556937970459,
    "confusion_matrix": [
      [
        28480,
        53
      ],
      [
        197,
        109
      ]
    ]
  },
  "Logistic Regression": {
    "accuracy": 0.9879676826519643,
    "precision_fraud": 0.0,
    "recall_fraud": 0.0,
    "f1_fraud": 0.0,
    "roc_auc": 0.8387539574060444,
    "confusion_matrix": [
      [
        28492,
        41
      ],
      [
        306,
        0
      ]
    ]
  },
  "Random Forest": {
    "accuracy": 0.996671174451264,
    "precision_fraud": 0.9772727272727273,
    "recall_fraud": 0.7026143790849673,
    "f1_fraud": 0.8174904942965779,
    "roc_auc": 0.9915208259029964,
    "confusion_matrix": [
      [
        28528,
        5
      ],
      [
        91,
        215
      ]
    ]
  }
}


## 7. Ajuste de hiperparâmetros (RandomizedSearchCV)

In [8]:
search = modeling.tune_random_forest(X_train_transform, y_train_transformed, config.RANDOM_SEED)

y_pred_tuned = search.predict(X_test_transform)
y_proba_tuned = search.predict_proba(X_test_transform)[:, 1]
tuned_metrics = evaluation.evaluate_model(y_test_transform, y_pred_tuned, y_proba_tuned)
tuned_metrics["best_params"] = search.best_params_
print(json.dumps(tuned_metrics, indent=2, default=str))

Fitting 5 folds for each of 20 candidates, totalling 100 fits


{
  "accuracy": 0.9963937723222026,
  "precision_fraud": 0.9719626168224299,
  "recall_fraud": 0.6797385620915033,
  "f1_fraud": 0.8,
  "roc_auc": 0.9973982653728086,
  "confusion_matrix": [
    [
      28527,
      6
    ],
    [
      98,
      208
    ]
  ],
  "best_params": {
    "n_estimators": "510",
    "min_samples_split": "14",
    "min_samples_leaf": "1",
    "max_depth": null
  }
}


## 8. Persistir métricas

In [9]:
evaluation.save_metrics(
    {"baseline": baseline_scores, "tuned_random_forest": tuned_metrics}, config.METRICS_OUTPUT_PATH
)

## 9. Inferência no holdout e geração da submissão

Os encoders e o `imputer` ajustados no treino são **reutilizados** aqui (nunca reajustados), garantindo que o holdout seja transformado de forma consistente com o treino.

In [10]:
df_holdout_raw = data.load_holdout_data()
df_holdout = df_holdout_raw.drop(columns=config.PII_AND_HIGH_CARDINALITY_COLUMNS)

features.add_datetime_features(df_holdout)
features.add_age_feature(df_holdout)
features.apply_label_encoders(df_holdout, encoders, config.CATEGORICAL_COLUMNS)

df_holdout_transform = imputer.transform(df_holdout)
predictions = search.predict(df_holdout_transform)

evaluation.save_submission(df_holdout_raw["trans_num"], predictions, config.SUBMISSION_OUTPUT_PATH)

---

Este notebook espelha `fraud_ml/` — mudanças no pipeline devem ser feitas lá (e em `MlFraude.py`), não diretamente aqui.